In [1]:
import xarray as xr
import pandas as pd
import os

In [2]:
motive1 = xr.open_dataset('tropics_mission/tropics_data/sg195_MOTIVE_2024_timeseries_p1.nc')
motive2 = xr.open_dataset('tropics_mission/tropics_data/sg195_MOTIVE_2024_timeseries_p2.nc')
display(motive2)

<xarray.Dataset> Size: 6MB
Dimensions:                                   (gps_info: 126,
                                               sg_data_point: 31487,
                                               trajectory: 42, dive: 42)
Coordinates:
    ctd_time                                  (sg_data_point) datetime64[ns] 252kB ...
    ctd_depth                                 (sg_data_point) float32 126kB ...
    latitude                                  (sg_data_point) float32 126kB ...
    longitude                                 (sg_data_point) float32 126kB ...
  * trajectory                                (trajectory) int32 168B 661 ......
Dimensions without coordinates: gps_info, sg_data_point, dive
Data variables: (12/66)
    gps_info_dive_number                      (gps_info) int32 504B ...
    sg_data_point_dive_number                 (sg_data_point) int32 126kB ...
    log_gps_time                              (gps_info) datetime64[ns] 1kB ...
    time                                      (sg_data_point) datetime64[ns] 252kB ...
    pressure                                  (sg_data_point) float32 126kB ...
    depth                                     (sg_data_point) float32 126kB ...
    ...                                        ...
    end_longitude                             (dive) float32 168B ...
    depth_avg_curr_east                       (dive) float32 168B ...
    depth_avg_curr_north                      (dive) float32 168B ...
    depth_avg_curr_qc                         (dive) |S1 42B ...
    latlong_qc                                (dive) |S1 42B ...
    glider                                    |S12 12B ...
Attributes: (12/47)
    project:                         MOTIVE 2024
    title:                           Physical, biological, and chemical data ...
    summary:                         SG195 MOTIVE 2024
    source:                          Seaglider SG195
    references:                      http://data.nodc.noaa.gov/accession/0092291
    processing_level:                1.12
    ...                              ...
    date_modified:                   2025-05-26T03:05:31Z
    uuid:                            6cc78a6e-39de-11f0-80b3-07c04838153a
    base_station_version:            3.0.2
    base_station_micro_version:      0
    quality_control_version:         1.12
    Conventions:                     CF-1.6

In [3]:
# two different files for the motive mission so we first concatenate them

keep_vars = ["time", "depth", "latitude", "longitude", "temperature", "salinity", "aanderaa4330_dissolved_oxygen", "aanderaa4330_instrument_dissolved_oxygen", "eng_wlbb2fl_sig470nm", "eng_wlbb2fl_sig700nm", "eng_wlbb2fl_sig695nm", "sg_data_point_dive_number"]

def strip(ds):
    ds = ds[keep_vars]             
    return ds

motive1_ds = strip(motive1)
motive2_ds = strip(motive2)

motive_ds = xr.concat([motive1_ds, motive2_ds], dim="sg_data_point", data_vars="all", coords="minimal", compat="override", join="override")

motive_ds = motive_ds.sortby("ctd_time")

#display(motive_ds)

In [4]:
time = motive_ds['time'][0].values

print(time)

2005-04-06T14:24:48.526000000


In [5]:
#Align time with sg_data_point and apply offset
adjusted_time = pd.to_datetime(motive_ds['time'].values) + pd.DateOffset(years=19, months=7, days=13)

motive_ds = motive_ds.assign_coords(time=('sg_data_point', adjusted_time))

motive_ds['PAR_470nm'] = motive_ds['eng_wlbb2fl_sig470nm']
motive_ds['particle_concentration_700nm'] = motive_ds['eng_wlbb2fl_sig700nm']
motive_ds['chlorophyll_695nm'] = motive_ds['eng_wlbb2fl_sig695nm']
motive_ds['dive_number'] = motive_ds['sg_data_point_dive_number']
motive_ds['dissolved_oxygen'] = motive_ds['aanderaa4330_dissolved_oxygen']
motive_ds['instrument_dissolved_oxygen'] = motive_ds['aanderaa4330_instrument_dissolved_oxygen']


# add metadata
motive_ds['PAR_470nm'].attrs['pre_cleaning_name'] = 'eng_wlbb2fl_sig470nm'
motive_ds['particle_concentration_700nm'].attrs['pre_cleaning_name'] = 'eng_wlbb2fl_sig700nm'
motive_ds['chlorophyll_695nm'].attrs['pre_cleaning_name'] = 'eng_wlbb2fl_sig695nm'
motive_ds['dive_number'] .attrs['pre_cleaning_name'] = 'sg_data_point_dive_number'
motive_ds['dissolved_oxygen'].attrs['pre_cleaning_name'] = 'aanderaa4330_dissolved_oxygen'
motive_ds['instrument_dissolved_oxygen'].attrs['pre_cleaning_name' = 'aanderaa4330_instrument_dissolved_oxygen'


#Select the relevant variables
new_motive_ds = motive_ds[['time', 'depth',  'latitude','longitude','temperature', 'salinity', 'dissolved_oxygen', 'instrument_dissolved_oxygen', 'PAR_470nm', 'particle_concentration_700nm', 'chlorophyll_695nm', 'dive_number']] 
print(new_motive_ds['time'][0].values)
display(new_motive_ds)
#Convert to DataFrame and save
new_motive_ds.to_dataframe().reset_index().to_csv('sg195_tropics_timeseries_cleaned.csv', index=False)
new_motive_ds.to_netcdf('sg195_tropics_timeseries_cleaned.nc')

2024-11-19T14:24:48.526000000


<xarray.Dataset> Size: 32MB
Dimensions:                       (sg_data_point: 501644)
Coordinates:
    time                          (sg_data_point) datetime64[ns] 4MB 2024-11-...
    latitude                      (sg_data_point) float32 2MB 0.7322 ... 10.07
    longitude                     (sg_data_point) float32 2MB -139.2 ... -144.3
    ctd_time                      (sg_data_point) datetime64[ns] 4MB 2005-04-...
    ctd_depth                     (sg_data_point) float32 2MB 0.3545 ... 0.1955
Dimensions without coordinates: sg_data_point
Data variables:
    depth                         (sg_data_point) float32 2MB 0.9209 ... 0.2202
    temperature                   (sg_data_point) float32 2MB 25.08 ... 27.57
    salinity                      (sg_data_point) float32 2MB nan nan ... nan
    dissolved_oxygen              (sg_data_point) float32 2MB nan nan ... nan
    instrument_dissolved_oxygen   (sg_data_point) float32 2MB nan nan ... nan
    PAR_470nm                     (sg_data_point) float32 2MB 43.0 47.0 ... nan
    particle_concentration_700nm  (sg_data_point) float32 2MB 109.0 ... nan
    chlorophyll_695nm             (sg_data_point) float32 2MB 117.0 ... nan
    dive_number                   (sg_data_point) int32 2MB 1 1 1 ... 703 703
Attributes: (12/47)
    project:                         MOTIVE 2024
    title:                           Physical, chemical, and biological data ...
    summary:                         SG195 MOTIVE 2024
    source:                          Seaglider SG195
    references:                      http://data.nodc.noaa.gov/accession/0092291
    processing_level:                1.12
    ...                              ...
    date_modified:                   2025-05-14T12:05:07Z
    uuid:                            73016ec2-30bf-11f0-80b3-07c04838153a
    base_station_version:            3.0.2
    base_station_micro_version:      0
    quality_control_version:         1.12
    Conventions:                     CF-1.6

In [6]:
# two different files for the motive mission so we first concatenate them
motive1_dac = xr.open_dataset('tropics_mission/tropics_data/sg195_MOTIVE_2024_timeseries_p1.nc')
motive2_dac = xr.open_dataset('tropics_mission/tropics_data/sg195_MOTIVE_2024_timeseries_p2.nc')

keep_vars = ["depth_avg_curr_east", "depth_avg_curr_north", "start_time", "end_time", "start_latitude", "end_latitude", "start_longitude", "end_longitude"]

motive1_ds_dac = strip(motive1_dac)
motive2_ds_dac = strip(motive2_dac)

motive_ds_dac = xr.concat([motive1_ds_dac, motive2_ds_dac], dim="dive", data_vars="all", coords="minimal", compat="override", join="override")

motive_ds_dac = motive_ds_dac.sortby("start_time")

#display(motive_ds_dac)


In [7]:
#Apply time apply offset (if needed)
adjusted_start = pd.to_datetime(motive_ds_dac['start_time'].values) + pd.DateOffset(years=19, months=7, days=13)
adjusted_end = pd.to_datetime(motive_ds_dac['end_time'].values) + pd.DateOffset(years=19, months=7, days=13)

motive_ds_dac = motive_ds_dac.assign_coords(start_time=('dive', adjusted_start))
motive_ds_dac = motive_ds_dac.assign_coords(end_time=('dive', adjusted_end))

motive_ds_dac['U_DAC'] = motive_ds_dac['depth_avg_curr_east']
motive_ds_dac['V_DAC'] = motive_ds_dac['depth_avg_curr_north']

# add metadata
motive_ds_dac['U_DAC'].attrs['pre_cleaning_name'] = 'depth_avg_curr_east'
motive_ds_dac['V_DAC'].attrs['pre_cleaning_name'] = 'depth_avg_curr_north'

#Select the relevant variables
new_motive_ds = motive_ds_dac[['U_DAC', 'V_DAC', 'start_time', 'end_time', 'start_latitude', 'end_latitude', 'start_longitude', 'end_longitude']]
display(new_motive_ds)

#Convert to DataFrame and save
new_motive_ds.to_dataframe().reset_index().to_csv('sg195_tropics_timeseries_DAC_cleaned.csv', index=False)
new_motive_ds.to_netcdf('sg195_tropics_timeseries_DAC_cleaned.nc')


<xarray.Dataset> Size: 28kB
Dimensions:          (dive: 703)
Coordinates:
    start_time       (dive) datetime64[ns] 6kB 2024-11-19T14:23:46 ... 2025-0...
    end_time         (dive) datetime64[ns] 6kB 2024-11-19T14:48:12 ... 2025-0...
Dimensions without coordinates: dive
Data variables:
    U_DAC            (dive) float32 3kB 0.5582 0.7924 ... -0.07873 -0.08706
    V_DAC            (dive) float32 3kB 0.5208 0.4539 0.3798 ... 0.0843 0.05894
    start_latitude   (dive) float32 3kB 0.7322 0.7448 0.7631 ... 10.04 10.06
    end_latitude     (dive) float32 3kB 0.7386 0.7517 0.7766 ... 10.06 10.07
    start_longitude  (dive) float32 3kB -139.2 -139.2 -139.1 ... -144.3 -144.3
    end_longitude    (dive) float32 3kB -139.2 -139.1 -139.1 ... -144.3 -144.3
Attributes: (12/47)
    project:                         MOTIVE 2024
    title:                           Physical, chemical, and biological data ...
    summary:                         SG195 MOTIVE 2024
    source:                          Seaglider SG195
    references:                      http://data.nodc.noaa.gov/accession/0092291
    processing_level:                1.12
    ...                              ...
    date_modified:                   2025-05-14T12:05:07Z
    uuid:                            73016ec2-30bf-11f0-80b3-07c04838153a
    base_station_version:            3.0.2
    base_station_micro_version:      0
    quality_control_version:         1.12
    Conventions:                     CF-1.6